# Demo · Knowledge base multi-fonte

**Use case** (route `/demo/multi-fonte` della chat): un tecnico fa una domanda che richiede **più fonti**. Foundry IQ pianifica le sotto-query e interroga insieme manuali (Blob), magazzino ricambi (MCP), policy (SharePoint remoto) e cataloghi web filtrati, restituendo contenuti estratti e riferimenti.

Il notebook legge endpoint e nomi dall'ambiente azd selezionato oppure dagli output del deployment ARM, esegue gli scenari della demo e registra un esito per ogni controllo (`PASS`, `WARN` non bloccante, `FAIL`, `SKIP`).

| Passo | Mostra |
| --- | --- |
| 1 | Composizione della KB: quali knowledge source e di che tipo. |
| 2 | Scenari della demo: quali fonti la KB ha interrogato per ogni domanda. |
| 3 | Dentro una risposta: pianificazione delle query e riferimenti per fonte. |
| 4 | Risposta finale sintetizzata con citazioni `[ref_id]` (opzionale). |

In [ ]:
CONFIG_SOURCE = "azd"             # "azd" oppure "arm"
AZD_ENVIRONMENT_NAME = ""          # vuoto = ambiente azd attualmente selezionato
SUBSCRIPTION_ID = ""               # opzionale; con azd deve coincidere con l'ambiente
ENVIRONMENT_NAME = "wpc-demo"      # usato solo per il deployment ARM
DEPLOYMENT_NAME = f"wpc-foundryiq-{ENVIRONMENT_NAME}"
SEARCH_API_VERSION = "2026-11-01-preview"
KB_NAME = ""                       # vuoto = nome dell'ambiente o wpc-foundryiq-kb
RUN_SYNTHESIS = True   # usa il deployment chat per comporre la risposta con citazioni

# Stesse domande della chat; "fonti" sono le categorie attese (Manuali, Ricambi, SharePoint, Web).
SCENARI = [
    {"titolo": "Rx1 · inceppamenti e motore di tensione", "fonti": ["Manuali", "Ricambi", "SharePoint"],
     "domanda": "Il Rx1 Digital Weave Integration Station ha inceppamenti del tessuto: cosa controllo secondo il manuale, abbiamo un Weave Tension Motor disponibile e le policy dicono di ripararlo o sostituirlo?"},
    {"titolo": "Reclaim Tank · membrane filtranti", "fonti": ["Manuali", "Ricambi", "SharePoint"],
     "domanda": "Devo cambiare le membrane filtranti della Graphene Vapor Reclaim Tank: ogni quanto va fatto, quante GRT-002 abbiamo a magazzino e come vanno smaltite quelle esauste?"},
    {"titolo": "Edge Node · alimentatore AD-60W-24", "fonti": ["Manuali", "Ricambi", "Web", "SharePoint"],
     "domanda": "L'Edge Control Node P4311 non si accende: come verifico l'alimentatore AD-60W-24, ne abbiamo a magazzino o un'alternativa, e quali modelli equivalenti vendono i distributori online?"},
    {"titolo": "Print Bed Heater DS-PBH-004 · alternative", "fonti": ["Ricambi", "Web"],
     "domanda": "Il Print Bed Heater DS-PBH-004 ha 45 giorni di lead time: ci sono alternative a magazzino o prodotti equivalenti sui cataloghi dei distributori?"},
]
SCENARIO_DETTAGLIO = 0   # indice dello scenario da analizzare nel passo 3

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: servono infra/, data/, agents/ e le app.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_azd_values(environment_name):
    return load_azd_env(environment_name, root=ROOT, override=True)


def load_outputs(deployment_name, subscription_id):
    deployment = az("deployment", "sub", "show", "--name", deployment_name, "--subscription", subscription_id)
    if not deployment:
        raise RuntimeError(f"Deployment ARM '{deployment_name}' non trovato nella subscription selezionata.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


def load_configuration(required):
    if CONFIG_SOURCE == "azd":
        out = load_azd_values(AZD_ENVIRONMENT_NAME)
        selected = out.get("AZURE_SUBSCRIPTION_ID")
        if not isinstance(selected, str) or not selected.strip():
            raise RuntimeError("AZURE_SUBSCRIPTION_ID mancante nell'ambiente azd selezionato.")
        if SUBSCRIPTION_ID and SUBSCRIPTION_ID.casefold() != selected.casefold():
            raise RuntimeError("SUBSCRIPTION_ID non corrisponde alla subscription dell'ambiente azd.")
        account = az("account", "show", "--subscription", selected)
        if account["id"].casefold() != selected.casefold():
            raise RuntimeError("Azure CLI ha restituito una subscription diversa da quella dell'ambiente azd.")
        source = f"ambiente azd {out.get('AZURE_ENV_NAME') or AZD_ENVIRONMENT_NAME or '(corrente)'}"
    elif CONFIG_SOURCE == "arm":
        account = az("account", "show", *(["--subscription", SUBSCRIPTION_ID] if SUBSCRIPTION_ID else []))
        out = load_outputs(DEPLOYMENT_NAME, account["id"])
        source = f"deployment ARM {DEPLOYMENT_NAME}"
    else:
        raise ValueError("CONFIG_SOURCE deve essere 'azd' oppure 'arm'.")
    missing = [key for key in ["AZURE_TENANT_ID", "AZURE_RESOURCE_GROUP", *required]
               if not isinstance(out.get(key), str) or not out[key].strip()]
    if missing:
        raise RuntimeError(f"Valori mancanti in {source}: {', '.join(missing)}.")
    if out["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
        raise RuntimeError("Il tenant della configurazione non corrisponde alla subscription selezionata in Azure CLI.")
    return out, account, source


required = ["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"]
if RUN_SYNTHESIS:
    required += ["FOUNDRYIQ_DEMO_OPENAI_ENDPOINT", "FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT"]
out, account, source = load_configuration(required)
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
RESOURCE_GROUP = out["AZURE_RESOURCE_GROUP"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID})")
print(f"Configurazione: {source} -> resource group {RESOURCE_GROUP}")

import pandas as pd
import requests
from azure.identity import AzureCliCredential
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
credential = AzureCliCredential(subscription=SUBSCRIPTION_ID)
SEARCH_SCOPE = "https://search.azure.com/.default"
checks = []


def record(test, status, detail=""):
    """Esito di un controllo: PASS, WARN (non bloccante), FAIL o SKIP."""
    checks.append({"controllo": test, "esito": status, "dettaglio": detail})
    print(f"[{status}] {test}" + (f" - {detail}" if detail else ""))

from urllib.parse import quote, unquote, urlsplit

SEARCH = out["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"].rstrip("/")
KB_NAME = KB_NAME or out.get("FOUNDRYIQ_DEMO_KB_NAME") or "wpc-foundryiq-kb"


def search_token():
    return credential.get_token(SEARCH_SCOPE).token


def search_get(path):
    response = requests.get(f"{SEARCH}/{path}?api-version={SEARCH_API_VERSION}", timeout=60,
                            headers={"Authorization": f"Bearer {search_token()}"})
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()


def retrieve(kb_name, body, user_token=None):
    """Chiama la retrieve API: l'identità dell'app autorizza la KB, il token utente (opzionale) filtra le fonti."""
    headers = {"Authorization": f"Bearer {search_token()}"}
    if user_token:
        headers["x-ms-query-source-authorization"] = user_token
    started = time.perf_counter()
    response = requests.post(f"{SEARCH}/knowledgebases/{quote(kb_name, safe='')}/retrieve?api-version={SEARCH_API_VERSION}",
                             headers=headers, json=body, timeout=180)
    elapsed = round((time.perf_counter() - started) * 1000)
    if not response.ok:
        raise RuntimeError(f"Retrieve {kb_name}: HTTP {response.status_code} {response.text[:600]}")
    result = response.json()
    result["_elapsed_ms"] = elapsed
    return result

In [ ]:
CATEGORIES = {"azureBlob": "Manuali", "searchIndex": "Manuali", "mcpServer": "Ricambi", "remoteSharePoint": "SharePoint", "web": "Web"}


def summarize_sources(result, kinds):
    """Aggrega l'attività della retrieve per knowledge source: query eseguite, risultati ed errori."""
    sources = {}
    for item in result.get("activity") or []:
        name = item.get("knowledgeSourceName")
        if not name:
            continue
        row = sources.setdefault(name, {"fonte": name, "categoria": CATEGORIES.get(kinds.get(name), kinds.get(name) or "?"),
                                        "query": 0, "risultati": 0, "errore": None})
        row["query"] += 1
        row["risultati"] += item.get("count") or 0
        if item.get("error"):
            row["errore"] = item["error"] if isinstance(item["error"], str) else json.dumps(item["error"])
    return list(sources.values())


def references_by_source(result):
    activity = {str(a["id"]): a for a in result.get("activity") or [] if a.get("id") is not None}
    rows = []
    for ref in result.get("references") or []:
        source = activity.get(str(ref.get("activitySource")), {})
        data = ref.get("sourceData") or {}
        location = ref.get("blobUrl") or ref.get("url") or ref.get("webUrl") or data.get("url") or ""
        title = ref.get("title") or data.get("title") or data.get("name") or unquote(location.rstrip("/").split("/")[-1])
        rows.append({"ref_id": ref.get("id"), "fonte": source.get("knowledgeSourceName"), "tipo": ref.get("type"),
                     "titolo": title or ref.get("docKey"), "score": ref.get("rerankerScore")})
    return rows

## 1 · Composizione della knowledge base

In [ ]:
kb = search_get(f"knowledgebases/{quote(KB_NAME, safe='')}")
if kb is None:
    raise RuntimeError(f"Knowledge base {KB_NAME} non trovata: esegui deploy/03-foundry-iq-agenti.ipynb.")
kinds = {}
rows = []
for ref in kb.get("knowledgeSources") or []:
    ks = search_get(f"knowledgesources/{quote(ref['name'], safe='')}") or {}
    kinds[ref["name"]] = ks.get("kind")
    rows.append({"knowledge source": ref["name"], "tipo": ks.get("kind"), "categoria": CATEGORIES.get(ks.get("kind"), "?"),
                 "descrizione": ks.get("description")})
display(pd.DataFrame(rows))
models = [(model.get("azureOpenAIParameters") or {}).get("deploymentId") for model in kb.get("models") or []]
print("Reasoning effort:", (kb.get("retrievalReasoningEffort") or {}).get("kind"), "| output:", kb.get("outputMode"),
      "| modello KB:", ", ".join(model for model in models if model) or "non indicato",
      "| sintesi chat:", out.get("FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT") if RUN_SYNTHESIS else "disattivata")
available = {r["categoria"] for r in rows}
record("KB con manuali, ricambi e web", "PASS" if {"Manuali", "Ricambi", "Web"} <= available else "FAIL", ", ".join(sorted(available)))
record("Policy SharePoint", "PASS" if "SharePoint" in available else "WARN",
       "remote SharePoint presente" if "SharePoint" in available else "non configurata: gli scenari useranno le altre fonti")

## 2 · Scenari della demo

Ogni domanda è inviata con la conversazione (`messages`): la KB pianifica le sotto-query. Il token dell'utente è inoltrato in `x-ms-query-source-authorization` perché SharePoint remoto interroga i documenti con i permessi dell'utente.

In [ ]:
user_token = search_token()
results, overview = [], []
for scenario in SCENARI:
    body = {"messages": [{"role": "user", "content": [{"type": "text", "text": scenario["domanda"]}]}], "includeActivity": True}
    try:
        result = retrieve(KB_NAME, body, user_token)
    except RuntimeError as exc:
        record(f"Retrieve · {scenario['titolo']}", "FAIL", str(exc)[:200])
        results.append(None)
        continue
    results.append(result)
    sources = summarize_sources(result, kinds)
    queried = {s["categoria"] for s in sources}
    expected = {c for c in scenario["fonti"] if c in available}
    errors = [s["fonte"] for s in sources if s["errore"]]
    overview.append({"scenario": scenario["titolo"], "fonti interrogate": ", ".join(sorted(queried)) or "-",
                     "query": sum(s["query"] for s in sources), "riferimenti": len(result.get("references") or []),
                     "ms": result["_elapsed_ms"], "errori": ", ".join(errors) or "-"})
    status = "FAIL" if not result.get("references") or errors else "PASS" if expected <= queried else "WARN"
    record(f"Multi-fonte · {scenario['titolo']}", status,
           f"attese {', '.join(sorted(expected))}; interrogate {', '.join(sorted(queried)) or 'nessuna'}")
display(pd.DataFrame(overview))

## 3 · Dentro una risposta

Le attività mostrano come la KB ha scomposto la domanda; i riferimenti dicono da quale fonte arriva ogni contenuto estratto.

In [ ]:
detail = results[SCENARIO_DETTAGLIO] if results else None
if detail is None:
    print("Scenario non disponibile (retrieve fallita).")
else:
    display(Markdown(f"**{SCENARI[SCENARIO_DETTAGLIO]['domanda']}**"))
    steps = []
    for item in detail.get("activity") or []:
        query = next((v["search"] for v in item.values() if isinstance(v, dict) and isinstance(v.get("search"), str)), None)
        steps.append({"passo": item.get("type"), "fonte": item.get("knowledgeSourceName") or "-", "query": query,
                      "risultati": item.get("count"), "ms": item.get("elapsedMs")})
    display(pd.DataFrame(steps))
    display(pd.DataFrame(references_by_source(detail)))

## 4 · Risposta con citazioni

Come la chat web: il modello compone la risposta **solo** dai contenuti estratti e cita i `ref_id`.

In [ ]:
if not RUN_SYNTHESIS or detail is None:
    record("Sintesi con citazioni", "SKIP", "RUN_SYNTHESIS=False o retrieve fallita")
else:
    from azure.identity import get_bearer_token_provider
    from openai import AzureOpenAI

    extracted = "\n".join(c.get("text", "") for m in detail.get("response") or [] for c in m.get("content") or [])
    client = AzureOpenAI(azure_endpoint=out["FOUNDRYIQ_DEMO_OPENAI_ENDPOINT"], api_version="2025-04-01-preview",
                         azure_ad_token_provider=get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default"))
    completion = client.chat.completions.create(model=out["FOUNDRYIQ_DEMO_CHAT_DEPLOYMENT"], max_completion_tokens=4000, messages=[
        {"role": "system", "content": "Sei l'assistente dei tecnici WPC. Rispondi in italiano usando solo i contenuti recuperati. "
                                      "Se non bastano dillo. Cita le fonti con il loro ref_id tra parentesi quadre, ad esempio [0]."},
        {"role": "user", "content": f"Contenuti recuperati da Foundry IQ:\n{extracted[:60000]}\n\nDomanda: {SCENARI[SCENARIO_DETTAGLIO]['domanda']}"}])
    answer = completion.choices[0].message.content or ""
    display(Markdown(answer))
    record("Sintesi con citazioni", "PASS" if re.search(r"\[\d+\]", answer) else "WARN", f"{len(answer)} caratteri")

In [ ]:
credential.close()
display(pd.DataFrame(checks))
failed = [c["controllo"] for c in checks if c["esito"] == "FAIL"]
if failed:
    raise AssertionError("Controlli falliti: " + ", ".join(failed))
print("Demo multi-fonte verificata. I WARN indicano fonti attese non interrogate dal planner, non errori.")